# Model Insights
This notebook houses all the work needed to generate the model insights displayed with the prediction tool itself. This includes: PDP's, SHAP values, Residual analysis.

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.inspection import partial_dependence
import os

In [2]:
# Loading the winning model pipeline and training/testing data
with open('../deployment/model_artifacts_002.pkl', 'rb') as f:
    artifacts = pickle.load(f)

best_model = artifacts['model']
train_df = pd.read_csv('../data/train_data.csv')
test_df = pd.read_csv('../data/test_data.csv')
X_train = train_df.drop(columns='Sold_Price')
X_test = test_df.drop(columns='Sold_Price')

In [3]:
# Selecting the continuous variables to visualize in PDP's
# Categorical vars/Target Encoded features are less interpretable on a PDP line chart
features_to_plot = [
    'Mileage', 'car_age', 'Engine_Displacement_L', 'mileage_per_year', "Gears"
]
pdp_results = []

# Fill NaNs temporarily so scikit-learn can calculate the percentiles
X_train_pdp = X_train.copy()
X_train_pdp['mileage_per_year'] = X_train_pdp['mileage_per_year'].fillna(X_train_pdp['mileage_per_year'].median())

print("Calculating Partial Dependency Plots...")
for feature in features_to_plot:
    pdp = partial_dependence(best_model, X_train_pdp, features=[feature], grid_resolution=50)
    
    # Accommodate scikit-learn version differences for the grid key
    grid_key = 'values' if 'values' in pdp else 'grid_values'
    grid_values = pdp[grid_key][0]
    
    # The model targets log-space, inverse transform to real dollars
    avg_predictions_dollars = np.expm1(pdp['average'][0])
    
    for val, pred in zip(grid_values, avg_predictions_dollars):
        pdp_results.append({
            'Feature': feature,
            'Feature_Value': val,
            'Predicted_Price': pred
        })

Calculating Partial Dependency Plots...


In [4]:
# Exporting PDP data to the frontend data directory
pdp_df = pd.DataFrame(pdp_results)
pdp_df.to_csv('../data/frontend_data/pdp_data.csv', index=False)
print("Saved PDP data to ../data/frontend_data/pdp_data.csv")

Saved PDP data to ../data/frontend_data/pdp_data.csv


In [5]:
# Moving to the residual data
df_residuals = X_test.copy()

# Original target in dollars
y_test_dollars = test_df['Sold_Price']

# Predictions in log space
y_pred_log = best_model.predict(X_test)

# Predictions in dollars 
y_pred_dollars = np.expm1(y_pred_log)

# Creating df with target and prediction
df_residuals['Sold_Price'] = y_test_dollars.values
df_residuals['Predicted_Price'] = y_pred_dollars

df_residuals.head()

,Make,Model,Mileage,Exterior Color,Interior Color,Gears,Engine_Displacement_L,flaw_count,2_keys_ind,owners_manual_ind,...,auction_month_5,auction_month_6,auction_month_7,auction_month_8,auction_month_9,auction_month_10,auction_month_11,auction_month_12,Sold_Price,Predicted_Price
0,BMW,3 Series,60000.0,10,1,4.0,2.5,1,1,0,...,True,False,False,False,False,False,False,False,10500.0,10405.302734
1,BMW,X7,5500.0,13,1,8.0,3.0,2,1,1,...,True,False,False,False,False,False,False,False,65500.0,79905.875000
2,Chevrolet,C7 Corvette,53900.0,11,10,7.0,6.2,1,1,0,...,True,False,False,False,False,False,False,False,46000.0,34208.414062
3,Bentley,Continental,32400.0,13,12,8.0,6.0,1,0,0,...,True,False,False,False,False,False,False,False,58111.0,37542.539062
4,Mazda,ND Miata,4600.0,5,1,6.0,2.0,1,1,1,...,True,False,False,False,False,False,False,False,25250.0,31234.658203


In [6]:
# Back-calculating the car's Year
if 'Year' not in df_residuals.columns and 'auction_year' in df_residuals.columns and 'car_age' in df_residuals.columns:
    df_residuals['Year'] = df_residuals['auction_year'] - df_residuals['car_age']

# Keeping only the columns needed for the frontend visualization to reduce file size
cols_to_keep = ['Make', 'Model', 'Year', 'Sold_Price', 'Predicted_Price']
available_cols = [c for c in cols_to_keep if c in df_residuals.columns]

os.makedirs('../data/frontend_data', exist_ok=True)
df_residuals[available_cols].to_csv('../data/frontend_data/residual_data.csv', index=False)

residual_data.csv exported successfully!
